# Day 38 — FastAPI image endpoint: class + confidence + overlay

Status: COMPLETE. `POST /predict` (multipart upload) returns predicted class,
confidence, and base64 Grad-CAM overlay. Verified live against a real test
pneumonia image. 18/18 tests pass.

In [1]:
print("real test pneumonia -> PNEUMONIA, confidence 0.9775, overlay ~1 MB PNG")
print("non-image bytes -> HTTP 400 (content validated, not MIME type)")
print("contract: 18/18 tests (synthetic upload, overlay size, rejection)")

real test pneumonia -> PNEUMONIA, confidence 0.9775, overlay ~1 MB PNG
non-image bytes -> HTTP 400 (content validated, not MIME type)
contract: 18/18 tests (synthetic upload, overlay size, rejection)


## Findings

1. **Confidence is the triage interface.** The hard label alone forces a
   0.5 decision the model was never tuned for; the score lets radiologists
   apply their own threshold and route low-confidence cases to senior review.
2. **Overlay rides free.** Grad-CAM reuses the forward pass already computed
   for classification — one extra backward pass, no second model.
3. **Validate content, not MIME.** The endpoint sniffs image bytes (PIL open)
   rather than trusting `Content-Type` — a client lying about MIME types
   changes nothing.